# Proyecto final — Red de transferencias de Ethereum

**Estructura de Datos — Universidad Industrial de Santander**

**Equipo:** Jhon Arciniegas, Pablo Álvarez y Emerson Ballesteros

---

Este notebook usa **un solo dataset**:

```
transferencias.parquet
```

El profesor lo sube y el notebook hace todo lo demás:

1. Analiza las 38,9 M de transferencias con DuckDB.
2. Genera una muestra conectada de USDC.
3. Ejecuta las cuatro tareas con estructuras de datos básicas.
4. Explica la teoría y la complejidad de cada algoritmo.

| Tarea | Pregunta | Algoritmo | Responsable |
|---|---|---|---|
| 1 | ¿Cuál es el camino más barato? | Bellman-Ford | Pablo Álvarez |
| 2 | ¿Qué nodos son más importantes? | Grado + PageRank | Jhon Arciniegas |
| 3 | ¿Qué grupos de nodos hay? | Louvain | Emerson Ballesteros |
| 4 | ¿Cuál es la red mínima? | Kruskal + Union-Find | Grupal |

> **Cómo ejecutar:** sube `transferencias.parquet`, ejecuta las celdas en
> orden y listo. No necesitas clonar el repositorio ni instalar módulos.

## Teoría: grafos y estructuras de datos

Un **grafo** tiene nodos y aristas. En este proyecto:

- **Nodo**: una dirección de Ethereum.
- **Arista**: una transferencia de una dirección a otra.
- **Peso**: la cantidad transferida.

El grafo es **dirigido** (A → B no es lo mismo que B → A) y **ponderado**.

### Estructuras de datos que usamos

| Estructura | Dónde se usa | Para qué |
|---|---|---|
| **Lista de adyacencia** (`dict` de listas) | T1, T2, T3 | recorrer los vecinos de un nodo |
| **Diccionario** | T1, T2, T3 | guardar distancias, puntajes y comunidades |
| **Conjunto** (`set`) | T2, T3 | nodos visitados y comunidades vecinas |
| **Cola** (`deque`) | T1, T3 | recorrer por niveles y encontrar componentes |
| **Lista de aristas** | T4 | ordenar las aristas por peso |
| **Union-Find** | T4 | detectar ciclos en Kruskal |

La idea es simple: **cada algoritmo usa la estructura que le permite hacer su trabajo sin recorrer el grafo de más**.

## 0. Capa de datos: el grafo

Esta celda define la estructura compartida: un grafo dirigido y ponderado.

- **Nodos**: direcciones Ethereum.
- **Aristas**: transferencias `(from_address, to_address, value)`.
- **Peso**: cantidad transferida.

El resto de celdas usan el mismo objeto `grafo`.

In [ ]:
"""
grafo.py
========
Carga del dataset y construcción del grafo.

En este proyecto modelamos las transferencias de un token ERC-20 de Ethereum
como un grafo DIRIGIDO y PONDERADO G = (V, E):

    - V (vértices): las direcciones (wallets, exchanges, contratos).
    - E (aristas): las transferencias de una dirección a otra.
    - w(e) (peso): la cantidad transferida.

La carga acepta **CSV** y **Parquet**. Para Parquet se usa DuckDB solo para
leer y muestrear el archivo; los algoritmos del proyecto siguen siendo Python
puro y reciben el grafo ya construido.

Solo usamos estructuras nativas de Python: diccionarios y listas.
"""

from collections import defaultdict, deque
from typing import Dict, List, Optional, Set, Tuple

# Lista de adyacencia: cada nodo apunta a una lista de (vecino, peso).
AdjList = Dict[str, List[Tuple[str, float]]]

# Nombres de columna que aceptamos para cada dato. Se comparan sin distinguir
# mayúsculas/minúsculas, así funciona con distintos exports del dataset.
COLUMNAS = {
    "from": ["from_address", "from", "sender", "from_addr", "fromaddress"],
    "to": ["to_address", "to", "receiver", "to_addr", "toaddress"],
    "value": ["value", "amount", "token_amount", "quantity", "val"],
    "token": ["token_address", "contract_address", "token", "tokenaddress"],
    "time": ["block_timestamp", "timestamp", "block_time", "date", "time"],
    "block": ["block_number", "block"],
}

DIRECCION_CERO = "0x0000000000000000000000000000000000000000"
DIRECCIONES_CERO = {DIRECCION_CERO, "0x0", "0x00"}


def _elegir_columnas(disponibles: list[str]) -> Dict[str, Optional[str]]:
    """Mapea cada dato lógico (from, to, value...) a su columna real.

    `disponibles` debe venir en minúsculas y en el orden del archivo; el
    resultado conserva el nombre real de la columna para poder citarlo en SQL.
    """
    return {
        clave: next((opcion for opcion in opciones if opcion in disponibles), None)
        for clave, opciones in COLUMNAS.items()
    }


def _sql_texto(valor: str) -> str:
    """Escapa un texto para usarlo dentro de una consulta SQL."""
    return "'" + str(valor).replace("'", "''") + "'"


def _sql_columna(columna: str) -> str:
    """Cita un identificador de columna para SQL."""
    return '"' + str(columna).replace('"', '""') + '"'


class Grafo:
    """Grafo dirigido y ponderado representado con una lista de adyacencia."""

    def __init__(self) -> None:
        # {origen: [(destino, peso), ...]}
        self.adj: AdjList = defaultdict(list)
        # Todos los vértices del grafo.
        self.nodos: Set[str] = set()
        # Lista de aristas como (origen, destino, peso).
        self.aristas: List[Tuple[str, str, float]] = []
        # {dirección: {"nombre": ..., "tipo": ...}} para mostrar resultados legibles.
        self.etiquetas: Dict[str, Dict[str, str]] = {}

    # ------------------------------------------------------------------
    # Carga de datos
    # ------------------------------------------------------------------

    def _agregar_arista(
        self,
        origen: str,
        destino: str,
        peso: float,
        limpiar: bool = True,
    ) -> bool:
        """Agrega una arista y devuelve si fue aceptada.

        Con `limpiar=True` se descartan transferencias inválidas para el
        análisis: dirección cero, auto-transferencias y valores no positivos.
        """
        origen = str(origen).strip().lower()
        destino = str(destino).strip().lower()
        peso = float(peso)

        if limpiar:
            if not origen or not destino:
                return False
            if origen == destino:
                return False
            if origen in DIRECCIONES_CERO or destino in DIRECCIONES_CERO:
                return False
            if peso <= 0:
                return False

        self.nodos.add(origen)
        self.nodos.add(destino)
        self.adj[origen].append((destino, peso))
        self.aristas.append((origen, destino, peso))
        return True

    def cargar_transferencias(
        self,
        ruta_csv: str,
        max_filas: Optional[int] = None,
        limpiar: bool = False,
    ) -> "Grafo":
        """Lee un CSV de transferencias y llena la lista de adyacencia.

        El CSV debe tener (al menos) columnas equivalentes a:
            from_address, to_address, value
        (se aceptan alias como `from`, `sender`, `to`, `receiver`, etc.).

        Parámetros:
            ruta_csv: ruta al archivo CSV.
            max_filas: opcional. Leer solo las primeras `max_filas` filas
                       (útil para probar con una muestra).
            limpiar: descartar dirección cero, auto-loops y valores <= 0.

        Retorna:
            self, para poder encadenar llamadas.
        """
        import csv

        with open(ruta_csv, newline="", encoding="utf-8") as archivo:
            lector = csv.DictReader(archivo)
            campos = lector.fieldnames or []
            disponibles = [c.lower() for c in campos]
            reales = {c.lower(): c for c in campos}
            mapa = _elegir_columnas(disponibles)

            faltan = [k for k in ("from", "to", "value") if not mapa[k]]
            if faltan:
                raise ValueError(
                    "No encontré las columnas necesarias: "
                    + ", ".join(faltan)
                    + ". Columnas disponibles: "
                    + ", ".join(campos)
                )

            col_from = reales[mapa["from"]]
            col_to = reales[mapa["to"]]
            col_value = reales[mapa["value"]]

            for i, fila in enumerate(lector):
                if max_filas is not None and i >= max_filas:
                    break
                self._agregar_arista(
                    fila[col_from],
                    fila[col_to],
                    fila[col_value],
                    limpiar=limpiar,
                )

        return self

    def cargar_parquet(
        self,
        ruta: str,
        max_filas: Optional[int] = None,
        muestra: Optional[int] = None,
        token: Optional[str] = None,
        desde: Optional[str] = None,
        hasta: Optional[str] = None,
        semilla: int = 42,
        limpiar: bool = False,
    ) -> "Grafo":
        """Lee un Parquet (o una muestra) usando DuckDB.

        DuckDB solo se encarga de leer, filtrar y muestrear; el grafo se
        construye igual que con el CSV, usando Python puro.

        Parámetros:
            ruta: ruta al archivo Parquet.
            max_filas: leer solo las primeras N filas.
            muestra: tomar una muestra aleatoria de N filas (recomendado para
                     el dataset completo).
            token: filtrar por `token_address`.
            desde / hasta: filtrar por `block_timestamp` (texto ISO) si existe.
            semilla: semilla de la muestra aleatoria.
            limpiar: descartar dirección cero, auto-loops y valores <= 0.

        Retorna:
            self, para poder encadenar llamadas.
        """
        try:
            import duckdb
        except ImportError as exc:  # pragma: no cover - depende del entorno
            raise SystemExit(
                "Falta duckdb para leer Parquet. Instálalo con:\n"
                "    pip install -r requirements-datos.txt"
            ) from exc

        con = duckdb.connect()
        try:
            campos = list(con.read_parquet(ruta).columns)
            disponibles = [c.lower() for c in campos]
            reales = {c.lower(): c for c in campos}
            mapa = _elegir_columnas(disponibles)

            faltan = [k for k in ("from", "to", "value") if not mapa[k]]
            if faltan:
                raise ValueError(
                    "No encontré las columnas necesarias: "
                    + ", ".join(faltan)
                    + ". Columnas disponibles: "
                    + ", ".join(campos)
                )

            select = (
                f"CAST({_sql_columna(reales[mapa['from']])} AS VARCHAR) AS from_address, "
                f"CAST({_sql_columna(reales[mapa['to']])} AS VARCHAR) AS to_address, "
                f"CAST({_sql_columna(reales[mapa['value']])} AS DOUBLE) AS value"
            )

            filtros: list[str] = []
            if token:
                if not mapa["token"]:
                    raise ValueError("Pediste `token` pero el Parquet no tiene columna de token.")
                filtros.append(
                    f"CAST({_sql_columna(reales[mapa['token']])} AS VARCHAR) = "
                    f"{_sql_texto(token)}"
                )
            if desde or hasta:
                if not mapa["time"]:
                    raise ValueError(
                        "Pediste una ventana de tiempo pero el Parquet no tiene "
                        "columna de tiempo (block_timestamp)."
                    )
                col_tiempo = _sql_columna(reales[mapa["time"]])
                if desde:
                    filtros.append(f"CAST({col_tiempo} AS VARCHAR) >= {_sql_texto(desde)}")
                if hasta:
                    filtros.append(f"CAST({col_tiempo} AS VARCHAR) <= {_sql_texto(hasta)}")

            consulta = f"SELECT {select} FROM read_parquet({_sql_texto(ruta)})"
            if filtros:
                consulta += " WHERE " + " AND ".join(filtros)

            if muestra is not None:
                n = int(muestra)
                consulta += f" USING SAMPLE reservoir({n} ROWS)"
                if semilla is not None:
                    consulta += f" REPEATABLE ({int(semilla)})"
            elif max_filas is not None:
                consulta += f" LIMIT {int(max_filas)}"

            try:
                filas = con.execute(consulta).fetchall()
            except Exception as exc:  # DuckDB antiguo sin REPEATABLE/reservoir
                mensaje = str(exc).lower()
                if muestra is not None and ("parser" in mensaje or "syntax" in mensaje):
                    base = consulta.split(" USING SAMPLE ")[0]
                    filas = con.execute(f"{base} LIMIT {int(muestra)}").fetchall()
                else:
                    raise
        finally:
            con.close()

        for origen, destino, peso in filas:
            self._agregar_arista(origen, destino, peso, limpiar=limpiar)

        return self

    def cargar(self, ruta: str, max_filas: Optional[int] = None, **opciones) -> "Grafo":
        """Carga transferencias detectando el formato por la extensión.

        - `.csv`  -> `cargar_transferencias`
        - `.parquet` / `.pq` -> `cargar_parquet`

        Las opciones específicas de Parquet (`muestra`, `token`, `desde`,
        `hasta`, `semilla`, `limpiar`) se pasan tal cual.
        """
        if str(ruta).lower().endswith((".parquet", ".pq")):
            return self.cargar_parquet(ruta, max_filas=max_filas, **opciones)

        sobrantes = set(opciones) - {"limpiar"}
        if sobrantes:
            raise ValueError(
                f"Las opciones {sorted(sobrantes)} solo aplican a archivos Parquet."
            )
        return self.cargar_transferencias(ruta, max_filas=max_filas, **opciones)

    def cargar_etiquetas(self, ruta_csv: str) -> "Grafo":
        """Lee un CSV con el nombre legible de cada dirección.

        El CSV debe tener las columnas: address, name, type.
        """
        import csv

        # Algunos exports de etiquetas vienen en latin-1 (nombres con tildes).
        filas = None
        for codificacion in ("utf-8", "latin-1"):
            try:
                with open(ruta_csv, newline="", encoding=codificacion) as archivo:
                    filas = list(csv.DictReader(archivo))
                break
            except UnicodeDecodeError:
                continue
        if filas is None:  # pragma: no cover - archivo binario inválido
            raise ValueError(f"No se pudo leer el CSV de etiquetas: {ruta_csv}")

        for fila in filas:
            addr = fila["address"].strip().lower()
            self.etiquetas[addr] = {
                "nombre": fila.get("name", "").strip(),
                "tipo": fila.get("type", "").strip(),
            }
        return self

    # ------------------------------------------------------------------
    # Utilidades
    # ------------------------------------------------------------------

    def conservar_componente_mayor(self) -> "Grafo":
        """Deja solo la componente conexa (no dirigida) más grande.

        Un muestreo aleatorio deja muchos nodos aislados. Quedarse con la
        componente más grande produce una red conectada y resultados
        interpretables, sin volver a leer el archivo.

        Retorna:
            self, para poder encadenar llamadas.
        """
        adyacencia: dict[str, set[str]] = defaultdict(set)
        for origen, destino, _ in self.aristas:
            adyacencia[origen].add(destino)
            adyacencia[destino].add(origen)

        visitados: Set[str] = set()
        mejor: Set[str] = set()
        for inicio in adyacencia:
            if inicio in visitados:
                continue
            componente: Set[str] = set()
            cola = deque([inicio])
            visitados.add(inicio)
            while cola:
                nodo = cola.popleft()
                componente.add(nodo)
                for vecino in adyacencia[nodo]:
                    if vecino not in visitados:
                        visitados.add(vecino)
                        cola.append(vecino)
            if len(componente) > len(mejor):
                mejor = componente

        if not mejor or len(mejor) == len(self.nodos):
            return self

        aristas = [
            (a, b, w) for a, b, w in self.aristas if a in mejor and b in mejor
        ]
        self.adj = defaultdict(list)
        self.nodos = set()
        self.aristas = []
        for origen, destino, peso in aristas:
            self._agregar_arista(origen, destino, peso, limpiar=False)
        return self

    def nombre(self, addr: str) -> str:
        """Nombre legible de un nodo, o la propia dirección si no está etiquetada."""
        return self.etiquetas.get(addr, {}).get("nombre", addr)

    def resumen(self) -> Dict[str, float]:
        """Estadísticas básicas: número de nodos, aristas, densidad y peso."""
        n = len(self.nodos)
        m = len(self.aristas)
        # Densidad de un grafo dirigido: |E| / (|V| * (|V| - 1))
        densidad = m / (n * (n - 1)) if n > 1 else 0.0
        peso_total = sum(w for _, _, w in self.aristas)
        return {
            "nodos": n,
            "aristas": m,
            "densidad": round(densidad, 4),
            "peso_total": round(peso_total, 2),
        }


def cargar_grafo(
    ruta: str,
    max_filas: Optional[int] = None,
    etiquetas: Optional[str] = None,
    limpiar: bool = True,
    **opciones,
) -> Grafo:
    """Carga el dataset UNA vez y devuelve el objeto compartido `Grafo`.

    Es la función que usa el notebook. Las cuatro tareas reciben
    `grafo.aristas` y construyen su propia estructura; no vuelven a leer el
    archivo.

    Por defecto limpia transferencias inválidas (dirección cero, autolazos y
    valores <= 0). Usa `limpiar=False` si quieres cargar el archivo tal cual.

    Ejemplo:
        grafo = cargar_grafo("data/sample/transferencias.csv")
    """
    opciones.setdefault("limpiar", limpiar)
    grafo = Grafo().cargar(ruta, max_filas=max_filas, **opciones)
    if etiquetas:
        grafo.cargar_etiquetas(etiquetas)
    return grafo

## 1. Conectar con Google Drive

Esta celda monta Google Drive para poder leer el dataset desde
`Mi unidad/ED2/`. En local se salta sola.

In [ ]:
# ============ MONTAR GOOGLE DRIVE (COLAB) ============
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive no disponible (normal si estás en local):", exc)

## Cargar el dataset

Sube **un solo archivo**:

```
transferencias.parquet
```

Puedes dejarlo en cualquiera de estas rutas:

```
/content/transferencias.parquet
Mi unidad/ED2/transferencias.parquet
data/procesado/transferencias.parquet
```

La celda busca el archivo, genera una muestra de USDC y construye el grafo
que usan las Tareas 1–4.

> Si estás en Colab, también puedes subirlo con **Archivo → Subir** o con:
>
> ```python
> # from google.colab import files
> # files.upload()
> ```

In [ ]:
# ============ RUTAS Y CARGA DEL DATASET ============
import os
from collections import defaultdict, deque

CARPETA_DRIVE = "/content/drive/MyDrive/ED2"
CARPETA_LOCAL = "data"

USDC = "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48"
CERO = "0x0000000000000000000000000000000000000000"


def buscar_archivo(*rutas):
    """Devuelve la primera ruta que exista, o None si no existe ninguna."""
    for ruta in rutas:
        if ruta and os.path.isfile(ruta):
            return ruta
    return None


RUTA_PARQUET = buscar_archivo(
    "/content/transferencias.parquet",
    f"{CARPETA_DRIVE}/transferencias.parquet",
    f"{CARPETA_LOCAL}/procesado/transferencias.parquet",
)

print("Parquet encontrado:", RUTA_PARQUET)

if RUTA_PARQUET is None:
    raise FileNotFoundError(
        "No encontré transferencias.parquet. Súbelo a /content/ o a "
        "Mi unidad/ED2/transferencias.parquet."
    )


def componente_mayor(aristas):
    """Conserva solo las aristas de la componente conexa más grande."""
    adyacencia = defaultdict(set)
    for origen, destino, _ in aristas:
        adyacencia[origen].add(destino)
        adyacencia[destino].add(origen)

    visitados = set()
    mejor = set()

    for inicio in adyacencia:
        if inicio in visitados:
            continue

        componente = set()
        cola = deque([inicio])
        visitados.add(inicio)

        while cola:
            nodo = cola.popleft()
            componente.add(nodo)
            for vecino in adyacencia[nodo]:
                if vecino not in visitados:
                    visitados.add(vecino)
                    cola.append(vecino)

        if len(componente) > len(mejor):
            mejor = componente

    return [
        (origen, destino, peso)
        for origen, destino, peso in aristas
        if origen in mejor and destino in mejor
    ]


def muestra_desde_parquet(ruta):
    """Genera una muestra de USDC desde el Parquet."""
    try:
        import duckdb
    except ImportError:
        import subprocess
        import sys
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "duckdb", "pyarrow"],
            check=True,
        )
        import duckdb

    con = duckdb.connect()
    con.execute("SET memory_limit='2GB'")
    con.execute("SET threads=2")

    filas = con.execute(
        """
        SELECT from_address, to_address, value
        FROM read_parquet(?)
        WHERE token_address = ?
        USING SAMPLE reservoir(65000 ROWS) REPEATABLE (42)
        """,
        [ruta, USDC],
    ).fetchall()

    aristas = []
    for origen, destino, valor in filas:
        origen = str(origen).lower()
        destino = str(destino).lower()
        valor = float(valor) / 1e6

        if not origen or not destino or origen == destino:
            continue
        if origen == CERO or destino == CERO:
            continue
        if valor <= 0:
            continue

        aristas.append((origen, destino, valor))

    return componente_mayor(aristas)


aristas = muestra_desde_parquet(RUTA_PARQUET)

grafo = Grafo()
for origen, destino, peso in aristas:
    grafo._agregar_arista(origen, destino, peso, limpiar=False)

print("Muestra generada desde el Parquet:")
print("  nodos   :", len(grafo.nodos))
print("  aristas :", len(grafo.aristas))
print("  resumen :", grafo.resumen())

## Tarea 1 — Bellman-Ford: camino de costo mínimo

**Responsable:** Pablo Álvarez

**Estructura de datos:** lista de adyacencia + diccionario de distancias.

El costo de un camino es la suma de los pesos de sus aristas. Bellman-Ford
relaja todas las aristas `|V|-1` veces y detecta ciclos negativos.

La implementación está en la celda siguiente y se prueba con la muestra.

In [ ]:
"""
bellman_ford.py
===============
Tarea 1 — Caminos de costo mínimo.

Problema: encontrar la ruta más "barata" entre dos direcciones del grafo,
donde el costo de una ruta es la suma de los pesos de sus aristas.

Regla del proyecto: NO se puede usar Dijkstra. Por eso usamos Bellman-Ford,
que además es capaz de detectar ciclos de peso negativo.

Idea del algoritmo:
    1. Empezamos con distancia 0 en el origen e "infinito" en el resto.
    2. Repetimos |V|-1 veces: por cada arista (u -> v, w), si
       dist[u] + w < dist[v], entonces actualizamos dist[v] = dist[u] + w.
       (a esto se le llama "relajar" una arista).
    3. Al terminar, dist[v] es la distancia mínima desde el origen a v.

Complejidad: O(|V| * |E|).
"""

from typing import Dict, List, Optional, Tuple

INF = float("inf")


def construir_adyacencia(aristas: List[Tuple[str, str, float]]) -> Dict[str, List[Tuple[str, float]]]:
    """Convierte la lista compartida de aristas en una lista de adyacencia.

    La lista de aristas viene de la carga única del dataset. Cada módulo
    construye su propia estructura, así no depende de la de otro.
    """
    from collections import defaultdict

    adj: Dict[str, List[Tuple[str, float]]] = defaultdict(list)
    for origen, destino, peso in aristas:
        adj[origen].append((destino, float(peso)))
    return dict(adj)


def bellman_ford(
    adj: Dict[str, List[Tuple[str, float]]],
    origen: str,
) -> Tuple[Dict[str, float], Dict[str, Optional[str]]]:
    """Calcula las distancias mínimas desde `origen` a todos los nodos.

    Parámetros:
        adj: lista de adyacencia {nodo: [(vecino, peso), ...]}.
        origen: nodo de partida.

    Retorna:
        (distancias, predecesores):
            - distancias[v] = costo mínimo desde `origen` hasta v
              (INF si v es inalcanzable).
            - predecesores[v] = nodo anterior a v en el camino mínimo
              (None para el origen).
    """
    # 1. Reunimos todos los nodos del grafo (incluye los que no tienen
    #    aristas salientes pero sí entrantes).
    nodos = set(adj.keys())
    for vecinos in adj.values():
        for dst, _ in vecinos:
            nodos.add(dst)

    # 2. Inicializamos distancias y predecesores.
    dist = {v: INF for v in nodos}
    dist[origen] = 0.0
    prev = {v: None for v in nodos}

    # 3. Relajamos todas las aristas |V|-1 veces.
    for _ in range(len(nodos) - 1):
        cambio = False
        for u in adj:
            for v, w in adj[u]:
                if dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w
                    prev[v] = u
                    cambio = True
        if not cambio:
            # Ya no mejora nada: terminamos antes de tiempo.
            break

    # 4. (Opcional) Una pasada extra para detectar ciclos de peso negativo.
    #    Con pesos positivos nunca ocurre, pero es una ventaja de Bellman-Ford.
    for u in adj:
        for v, w in adj[u]:
            if dist[u] + w < dist[v]:
                raise ValueError("Se detectó un ciclo de peso negativo.")

    return dist, prev


def reconstruir_camino(
    prev: Dict[str, Optional[str]],
    origen: str,
    destino: str,
) -> List[str]:
    """Reconstruye el camino mínimo desde `origen` hasta `destino`.

    Usa el diccionario de predecesores que devuelve `bellman_ford`.

    Retorna:
        Lista de nodos [origen, ..., destino], o [] si no hay camino.
    """
    camino: List[str] = []
    actual: Optional[str] = destino

    # Retrocedemos desde el destino hasta el origen siguiendo `prev`.
    while actual is not None:
        camino.append(actual)
        actual = prev.get(actual)

    camino.reverse()
    return camino if camino and camino[0] == origen else []


def bellman_ford_desde_aristas(
    aristas: List[Tuple[str, str, float]],
    origen: str,
) -> Tuple[Dict[str, float], Dict[str, Optional[str]]]:
    """Igual que `bellman_ford`, pero recibe la lista compartida de aristas."""
    return bellman_ford(construir_adyacencia(aristas), origen)


# ============ DEMO TAREA 1 ============
from collections import Counter

salidas = Counter(origen for origen, _, _ in grafo.aristas)
etiquetados = sorted(
    (n for n in salidas if n in grafo.etiquetas),
    key=salidas.get, reverse=True,
)
origen = etiquetados[0] if etiquetados else max(salidas, key=salidas.get)

distancias, previos = bellman_ford_desde_aristas(grafo.aristas, origen)
alcanzables = sorted(
    (d, nodo) for nodo, d in distancias.items()
    if d != float("inf") and nodo != origen
)
inalcanzables = sum(1 for d in distancias.values() if d == float("inf"))

print(f"Origen: {grafo.nombre(origen)}")
print(f"Nodos alcanzables: {len(alcanzables):,}")
print(f"Nodos inalcanzables: {inalcanzables:,}\n")
print("Los 10 destinos más baratos:")
for d, destino in alcanzables[:10]:
    camino = reconstruir_camino(previos, origen, destino)
    nombres = " -> ".join(grafo.nombre(n) for n in camino)
    print(f"  {grafo.nombre(destino):<16} costo={d:>14,.2f}  ({nombres})")

## Tarea 2 — Centralidad y PageRank

**Responsable:** Jhon Arciniegas

**Estructura de datos:** lista de adyacencia + diccionario de puntajes.

- **Grado**: número de conexiones de un nodo.
- **PageRank**: importancia según quién apunta a quién.

PageRank se calcula con iteración de potencias y factor de amortiguación
`d = 0.85`.

La gráfica del top 5 y las validaciones de esta sección fueron aportadas por
Jhon Arciniegas (T2).

In [ ]:
from typing import Dict, List, Optional, Set, Tuple


def centralidad_grado(adj, nodos):
    """Centralidad de grado: C(v) = (vecinos distintos de v) / (|V| - 1).

    Cuenta direcciones DISTINTAS con las que v se conecta (envia o recibe),
    sin importar cuántas transferencias haya entre ellas. Va de 0 a 1
    """
    n = len(nodos)
    if n <= 1:
        return {v: 0.0 for v in nodos}
    vecinos = {v: set() for v in nodos}
    for u, destinos in adj.items():
        for v, _ in destinos:
            if u != v and u in vecinos and v in vecinos:
                vecinos[u].add(v)
                vecinos[v].add(u)
    return {v: len(vecinos[v]) / (n - 1) for v in nodos}


def pagerank(
    adj: Dict[str, List[Tuple[str, float]]],
    nodos: Set[str],
    amortiguacion: float = 0.85,
    max_iter: int = 100,
    tol: float = 1e-6,
) -> Dict[str, float]:
    """Calcula PageRank con el método de iteración de potencias.

    parametros:
        adj: lista de adyacencia
        nodos: conjunto de todos los vertices
        amortiguacion: factor de amortiguación d (0.85 por defecto)
        max_iter: máximo número de iteraciones
        tol: tolerancia de convergencia

    Retorna:
        {nodo: score} con scores que suman aproximadamente 1
    """
    n = len(nodos)
    if n == 0:
        return {}

    lista = list(nodos)

    # Puntaje inicial: todos los nodos igual de importantes (1/N).
    pr = {v: 1.0 / n for v in lista}

    # Grado de salida de cada nodo.
    grado_salida = {v: len(adj.get(v, [])) for v in lista}

    # Lista de adyacencia INVERSA: quién apunta a cada nodo.
    # Esto hace el cálculo más directo y eficiente.
    entrantes = {v: [] for v in lista}
    for u, vecinos in adj.items():
        for v, _ in vecinos:
            if v in entrantes:
                entrantes[v].append(u)

    for _ in range(max_iter):
        base = (1.0 - amortiguacion) / n

        # Los nodos "sumidero" (sin aristas de salida) no reparten su
        # importancia a nadie. Para que los puntajes sigan sumando 1, su masa
        # se reparte por igual entre todos los nodos.
        masa_sumidero = sum(pr[v] for v in lista if grado_salida[v] == 0)

        nuevo = {}
        for v in lista:
            suma = 0.0
            for u in entrantes[v]:
                if grado_salida[u] > 0:
                    suma += pr[u] / grado_salida[u]
            nuevo[v] = base + amortiguacion * (suma + masa_sumidero / n)

        # Medimos cuánto cambió el vector para saber si ya convergió.
        delta = sum(abs(nuevo[v] - pr[v]) for v in lista)
        pr = nuevo
        if delta < tol:
            break

    return pr


def top_n(
    puntajes: Dict[str, float],
    n: int = 10,
    etiquetas: Optional[Dict[str, Dict[str, str]]] = None,
) -> List[Tuple[str, float, str]]:
    """devuelve los `n` nodos con mayor puntaje, ordenados de mayor a menor

    retorna:
        lista de tuplas (direccion, puntaje, nombre) para mayor orden
    """
    etiquetas = etiquetas or {}

    # Orden: mayor puntaje primero; si empatan, por direccion (alfabetico)
    ordenados = sorted(puntajes.items(), key=lambda par: (-par[1], par[0]))[:n]

    resultado = []
    for addr, score in ordenados:
        # Si la etiqueta no existe o su nombre esta vacio, usamos la direccion
        nombre = etiquetas.get(addr, {}).get("nombre") or addr
        resultado.append((addr, score, nombre))
    return resultado


def construir_adyacencia(aristas: List[Tuple[str, str, float]]) -> Dict[str, List[Tuple[str, float]]]:
    """Convierte la lista compartida de aristas en lista de adyacencia."""
    adj: Dict[str, List[Tuple[str, float]]] = {}
    for origen, destino, peso in aristas:
        adj.setdefault(origen, []).append((destino, float(peso)))
    return adj


def nodos_de_aristas(aristas: List[Tuple[str, str, float]]) -> Set[str]:
    """Devuelve todos los nodos que aparecen en la lista de aristas."""
    nodos: Set[str] = set()
    for origen, destino, _ in aristas:
        nodos.add(origen)
        nodos.add(destino)
    return nodos


def centralidad_grado_desde_aristas(
    aristas: List[Tuple[str, str, float]],
) -> Dict[str, float]:
    """Centralidad de grado a partir de la lista compartida de aristas."""
    return centralidad_grado(construir_adyacencia(aristas), nodos_de_aristas(aristas))


def pagerank_desde_aristas(
    aristas: List[Tuple[str, str, float]],
    **opciones,
) -> Dict[str, float]:
    """PageRank a partir de la lista compartida de aristas."""
    return pagerank(
        construir_adyacencia(aristas),
        nodos_de_aristas(aristas),
        **opciones,
    )


# ============ DEMO TAREA 2 ============
import matplotlib.pyplot as plt

cent_grado = centralidad_grado_desde_aristas(grafo.aristas)
pr = pagerank_desde_aristas(grafo.aristas)

print("Top 10 por grado:")
for _, score, nombre in top_n(cent_grado, n=10, etiquetas=grafo.etiquetas):
    print(f"  {nombre:<18} {score:.4f}")

print("\nTop 10 por PageRank:")
for _, score, nombre in top_n(pr, n=10, etiquetas=grafo.etiquetas):
    print(f"  {nombre:<18} {score:.4f}")

# Gráfica y validaciones aportadas por Jhon Arciniegas (T2).
top_pr = top_n(pr, n=5, etiquetas=grafo.etiquetas)
nombres_pr = [nombre for _, _, nombre in top_pr]
scores_pr = [score for _, score, _ in top_pr]

plt.figure(figsize=(7, 4))
plt.barh(nombres_pr[::-1], scores_pr[::-1], color="steelblue")
plt.xlabel("PageRank")
plt.title("Top 5 direcciones por PageRank")
plt.tight_layout()
plt.show()

# Validaciones de centralidad (mismas que tests/test_centralidad.py).
assert centralidad_grado({"A": []}, {"A"})["A"] == 0.0
assert pagerank({}, set()) == {}
assert top_n({"A": 0.5, "B": 0.3}, n=2)[0][0] == "A"
print("Validaciones de centralidad: OK")

## Tarea 3 — Louvain: detección de comunidades

**Responsable:** Emerson Ballesteros

### Antes del código: ¿qué es una comunidad?

Imagina cuatro direcciones:

```
Ana   <--> Beto     (se transfieren mucho)
Carla <--> Dani     (se transfieren mucho)
Beto  -->  Carla    (se transfieren poquito)
```

Hay dos grupos naturales: `{Ana, Beto}` y `{Carla, Dani}`. Louvain intenta
encontrar esos grupos automáticamente, sin que le digamos cuántos hay.

### ¿Qué mide la modularidad Q?

Q compara las aristas que quedan **dentro** de los grupos con las que
aparecerían si las conexiones fueran al azar. Si hay más conexiones internas
de las esperadas, Q sube.

$$Q = \sum_c \left[\frac{L_c}{m} - \left(\frac{K_c}{2m}\right)^2\right]$$

- `L_c`: peso de las aristas internas de la comunidad `c`.
- `K_c`: suma de grados de los nodos de `c`.
- `m`: peso total del grafo.
- `Q > 0.3` sugiere una estructura de comunidades clara.
- `Q = 0` significa que la partición no es mejor que el azar.

### Paso a paso del algoritmo

1. Cada nodo empieza en su propia comunidad.
2. Se recorre la lista de nodos en orden aleatorio.
3. Para cada nodo, se prueba moverlo a la comunidad de cada vecino.
4. Se acepta el movimiento solo si Q aumenta.
5. Cuando una pasada completa no cambia nada, el algoritmo termina.

### Cómo leer el código

| Función | Qué hace |
|---|---|
| `simetrizar` | convierte el grafo dirigido en no dirigido |
| `calcular_grados` | suma los pesos de las aristas de cada nodo |
| `modularidad` | calcula Q de una partición |
| `louvain` | hace las pasadas y mueve los nodos |
| `desde_aristas` | construye el grafo no dirigido desde `grafo.aristas` |
| `louvain_desde_aristas` | atajo: construye el grafo y ejecuta Louvain |
| `renumerar_comunidades` | cambia los IDs por 0, 1, 2, ... para leerlos mejor |

### Qué mirar en los resultados

- Los **IDs de comunidad son arbitrarios**: la comunidad 0 no es mejor que la 5.
- Lo importante es **Q**, el **tamaño** de las comunidades y **qué exchanges**
  quedan juntos.
- Louvain es **heurístico**: dos semillas pueden dar particiones parecidas
  pero no idénticas.

### Limitaciones y posibles mejoras

- Esta versión recalcula Q completo en cada prueba: es fácil de leer, pero lenta.
- No incluye la fase de agregación de Louvain.
- Puede sufrir el límite de resolución: comunidades muy pequeñas pueden pasar
  desapercibidas.
- Mejoras: calcular ΔQ incremental, agregar la fase de agregación, probar
  varias semillas y comparar con Label Propagation o `python-louvain`.

In [ ]:
"""
comunidades.py
==============
Tarea 3 — Detección de comunidades con Louvain (versión didáctica).

Responsable: Emerson Ballesteros.

¿Qué es una comunidad?
----------------------
Una comunidad es un grupo de nodos que interactúa mucho entre sí y poco con
el resto de la red. Por ejemplo, los exchanges centralizados y sus usuarios
suelen formar un grupo, y los DEX con sus usuarios otro.

¿Cómo se mide si una partición es buena?
----------------------------------------
Con la **modularidad Q**. La idea es comparar las aristas que quedan *dentro*
de los grupos con las que aparecerían si las conexiones fueran al azar:

    Q = (1 / 2m) * suma_{i,j} [ A_ij - k_i*k_j/(2m) ] * delta(c_i, c_j)

    - A_ij: peso de la arista entre i y j.
    - k_i: grado (suma de pesos) del nodo i.
    - m: peso total del grafo.
    - delta(c_i, c_j): 1 si i y j están en la misma comunidad, 0 si no.

Para no recorrer todos los pares de nodos, el código usa una fórmula
equivalente que se calcula por comunidad:

    Q = suma_c [ L_c / m - (K_c / (2m))^2 ]

    - L_c: peso de las aristas internas de la comunidad c.
    - K_c: suma de grados de los nodos de la comunidad c.

Valores de Q por encima de ~0.3 sugieren una buena estructura de comunidades.

Algoritmo (fase principal de Louvain)
-------------------------------------
1. Cada nodo empieza en su propia comunidad.
2. Se repite hasta que nada cambie:
   - Se recorren los nodos en orden aleatorio.
   - Para cada nodo, se prueba moverlo a la comunidad de cada vecino.
   - Se queda con el movimiento que más aumente Q.
3. Se devuelve la comunidad final de cada nodo.

Esta es una versión **sencilla y fácil de leer**: recalcula Q completo en cada
prueba. La versión de producción usa una fórmula incremental (ΔQ) y una fase
de agregación. Al final del archivo hay una lista de posibles mejoras.
"""

import random
from collections import defaultdict
from typing import Dict, List, Optional, Set, Tuple

# Tipos con nombres claros para que el código se lea mejor.
Adyacencia = Dict[str, List[Tuple[str, float]]]
Comunidades = Dict[str, int]


def simetrizar(adj: Adyacencia) -> Adyacencia:
    """Convierte un grafo dirigido en un grafo NO dirigido.

    Si existe la arista u -> v con peso w, se crea la arista no dirigida
    {u, v} con peso w. Si existen u -> v y v -> u, los pesos se suman.

    Para Louvain trabajamos con el grafo no dirigido porque una comunidad no
    tiene dirección: interesa si dos nodos están conectados, no en qué sentido.
    """
    # Paso 1: juntar los pesos de cada par no ordenado {a, b}.
    pesos_por_par: Dict[Tuple[str, str], float] = defaultdict(float)

    for origen, vecinos in adj.items():
        for destino, peso in vecinos:
            # sorted((origen, destino)) hace que (A, B) y (B, A) sean el
            # mismo par. Así no contamos dos veces la misma conexión.
            a, b = sorted((origen, destino))
            pesos_por_par[(a, b)] += peso

    # Paso 2: construir la lista de adyacencia no dirigida.
    adj_no_dirigida: Adyacencia = defaultdict(list)

    for (a, b), peso in pesos_por_par.items():
        # La arista existe en los dos sentidos.
        adj_no_dirigida[a].append((b, peso))
        adj_no_dirigida[b].append((a, peso))

    return dict(adj_no_dirigida)


def calcular_grados(adj_no_dirigida: Adyacencia) -> Dict[str, float]:
    """Calcula el grado ponderado de cada nodo.

    El grado de un nodo es la suma de los pesos de todas sus aristas.
    En un grafo no dirigido, cada arista aporta su peso a los dos extremos.
    """
    grados: Dict[str, float] = {}

    for nodo, vecinos in adj_no_dirigida.items():
        suma_de_pesos = 0.0
        for _, peso in vecinos:
            suma_de_pesos += peso
        grados[nodo] = suma_de_pesos

    return grados


def modularidad(
    adj_no_dirigida: Adyacencia,
    comunidades: Comunidades,
) -> float:
    """Calcula la modularidad Q de una partición.

    Se usa la fórmula por comunidad:

        Q = suma_c [ L_c / m - (K_c / (2m))^2 ]

    Parámetros:
        adj_no_dirigida: lista de adyacencia del grafo no dirigido.
        comunidades: diccionario {nodo: id_de_comunidad}.

    Retorna:
        Q, un número entre -0.5 y 1.0.
    """
    # Paso 1: grado de cada nodo.
    grados = calcular_grados(adj_no_dirigida)

    # Paso 2: m = peso total del grafo.
    # La suma de todos los grados cuenta cada arista dos veces, por eso / 2.
    suma_de_grados = 0.0
    for grado in grados.values():
        suma_de_grados += grado
    peso_total = suma_de_grados / 2.0

    if peso_total == 0.0:
        return 0.0

    # Paso 3: K_c = suma de grados de los nodos de cada comunidad.
    grado_por_comunidad: Dict[int, float] = defaultdict(float)
    for nodo, grado in grados.items():
        comunidad = comunidades[nodo]
        grado_por_comunidad[comunidad] += grado

    # Paso 4: L_c = peso de las aristas internas de cada comunidad.
    # Cada arista no dirigida se cuenta una sola vez. Para lograrlo, solo se
    # suma cuando nodo < vecino (o una vez si es un auto-loop).
    interno_por_comunidad: Dict[int, float] = defaultdict(float)

    for nodo, vecinos in adj_no_dirigida.items():
        for vecino, peso in vecinos:
            if comunidades[nodo] != comunidades[vecino]:
                continue  # la arista sale de la comunidad: no es interna

            if nodo < vecino:
                interno_por_comunidad[comunidades[nodo]] += peso
            elif nodo == vecino:
                # Un auto-loop suma dos veces al grado; para la parte interna
                # se cuenta una sola vez.
                interno_por_comunidad[comunidades[nodo]] += peso / 2.0

    # Paso 5: aplicar la fórmula para cada comunidad.
    q = 0.0
    for comunidad, grado_total in grado_por_comunidad.items():
        interno = interno_por_comunidad.get(comunidad, 0.0)
        parte_interna = interno / peso_total
        parte_esperada = (grado_total / (2.0 * peso_total)) ** 2
        q += parte_interna - parte_esperada

    return q


def louvain(
    adj_no_dirigida: Adyacencia,
    semilla: int = 42,
    max_pasadas: int = 20,
) -> Comunidades:
    """Detecta comunidades maximizando la modularidad.

    Es la fase principal de Louvain:

    - Cada nodo empieza solo en su comunidad.
    - En cada pasada, se prueba mover cada nodo a la comunidad de sus vecinos.
    - Se acepta el movimiento solo si Q aumenta.
    - Se termina cuando una pasada completa no cambia nada.

    Parámetros:
        adj_no_dirigida: grafo no dirigido.
        semilla: semilla para que el resultado sea reproducible.
        max_pasadas: máximo de pasadas completas.

    Retorna:
        {nodo: id_de_comunidad}.
    """
    # random.Random(semilla) crea un generador propio. Así no cambiamos el
    # generador global de Python y el resultado es reproducible.
    generador = random.Random(semilla)

    nodos = list(adj_no_dirigida.keys())

    # Al inicio, cada nodo está en su propia comunidad.
    comunidades: Comunidades = {}
    for identificador, nodo in enumerate(nodos):
        comunidades[nodo] = identificador

    for numero_pasada in range(max_pasadas):
        hubo_cambio = False

        # Orden aleatorio para no favorecer siempre a los mismos nodos.
        generador.shuffle(nodos)

        for nodo in nodos:
            comunidad_actual = comunidades[nodo]
            mejor_comunidad = comunidad_actual
            mejor_q = modularidad(adj_no_dirigida, comunidades)

            # Comunidades de los vecinos. Usamos un set para no probar dos
            # veces la misma comunidad.
            comunidades_vecinas: Set[int] = set()
            for vecino, _ in adj_no_dirigida.get(nodo, []):
                comunidades_vecinas.add(comunidades[vecino])

            # Probamos cada comunidad vecina (en orden fijo para que sea
            # reproducible).
            for comunidad_vecina in sorted(comunidades_vecinas):
                if comunidad_vecina == comunidad_actual:
                    continue

                # Mover temporalmente el nodo.
                comunidades[nodo] = comunidad_vecina
                q_con_movimiento = modularidad(adj_no_dirigida, comunidades)

                if q_con_movimiento > mejor_q:
                    mejor_q = q_con_movimiento
                    mejor_comunidad = comunidad_vecina

                # Deshacer el movimiento antes de probar el siguiente.
                comunidades[nodo] = comunidad_actual

            # Si encontramos una comunidad mejor, dejamos el nodo allí.
            if mejor_comunidad != comunidad_actual:
                comunidades[nodo] = mejor_comunidad
                hubo_cambio = True

        # Si nadie se movió, ya no tiene sentido seguir.
        if not hubo_cambio:
            break

    return comunidades


def resumen_comunidades(
    comunidades: Comunidades,
    etiquetas: Optional[Dict[str, Dict[str, str]]] = None,
) -> Dict[int, List[str]]:
    """Agrupa los nodos por comunidad y devuelve nombres legibles.

    Parámetros:
        comunidades: {nodo: id_de_comunidad}.
        etiquetas: opcional, {dirección: {"nombre": ...}}.

    Retorna:
        {id_de_comunidad: [nombres de sus nodos]}.
    """
    grupos: Dict[int, List[str]] = defaultdict(list)

    for nodo, comunidad in comunidades.items():
        nombre = nodo
        if etiquetas is not None and nodo in etiquetas:
            nombre = etiquetas[nodo].get("nombre", nodo)
        grupos[comunidad].append(nombre)

    return dict(grupos)


def renumerar_comunidades(comunidades: Comunidades) -> Comunidades:
    """Cambia los IDs de comunidad por 0, 1, 2, ... en orden de aparición.

    Louvain devuelve IDs que son el índice original de un nodo (por ejemplo,
    300 o 692). Eso es correcto, pero para leer los resultados es más cómodo
    que las comunidades se llamen 0, 1, 2, ...

    No cambia la partición: solo cambia el nombre de cada grupo.
    """
    mapa_de_ids: Dict[int, int] = {}
    comunidades_nuevas: Comunidades = {}

    for nodo, id_viejo in comunidades.items():
        if id_viejo not in mapa_de_ids:
            mapa_de_ids[id_viejo] = len(mapa_de_ids)
        comunidades_nuevas[nodo] = mapa_de_ids[id_viejo]

    return comunidades_nuevas


def desde_aristas(aristas: List[Tuple[str, str, float]]) -> Adyacencia:
    """Construye el grafo no dirigido a partir de la lista compartida de aristas.

    La lista `grafo.aristas` viene de la carga única del dataset. Cada módulo
    recibe esa lista y construye su propia estructura; así no depende de la
    estructura de otro módulo.

    Si hay transferencias en ambos sentidos, sus pesos se suman.
    """
    # Primero construimos el grafo dirigido tal como viene en las aristas.
    adj_dirigida: Adyacencia = {}

    for origen, destino, peso in aristas:
        if origen not in adj_dirigida:
            adj_dirigida[origen] = []
        adj_dirigida[origen].append((destino, float(peso)))

    # Después lo convertimos a no dirigido.
    return simetrizar(adj_dirigida)


def louvain_desde_aristas(
    aristas: List[Tuple[str, str, float]],
    semilla: int = 42,
    max_pasadas: int = 20,
) -> Comunidades:
    """Atajo: construye el grafo no dirigido y ejecuta Louvain.

    Es la función que usa el notebook. Recibe directamente `grafo.aristas`.
    """
    adj_no_dirigida = desde_aristas(aristas)
    return louvain(
        adj_no_dirigida,
        semilla=semilla,
        max_pasadas=max_pasadas,
    )


# ----------------------------------------------------------------------
# Posibles mejoras (para cuando el proyecto crezca)
# ----------------------------------------------------------------------
# 1. Calcular el cambio de modularidad ΔQ con una fórmula incremental, en
#    vez de recalcular Q completo en cada prueba. Eso hace el algoritmo mucho
#    más rápido en grafos grandes.
# 2. Agregar la segunda fase de Louvain: convertir cada comunidad en un
#    super-nodo y repetir el proceso. Es la versión completa del paper.
# 3. Agregar el parámetro de resolución γ, que permite encontrar comunidades
#    más grandes o más pequeñas.
# 4. Probar varias semillas y quedarse con la partición de mayor Q.
# 5. Comparar con Label Propagation y con `python-louvain` usando la misma
#    muestra y las mismas etiquetas de exchanges.
# 6. Para grafos de millones de aristas, usar arreglos/CSR en vez de
#    diccionarios de Python.


# ============ TAREA 3: LOUVAIN ============
# Paso 1: construir el grafo no dirigido.
adj_no_dirigido = desde_aristas(grafo.aristas)

cantidad_nodos = len(adj_no_dirigido)
cantidad_aristas = 0
for vecinos in adj_no_dirigido.values():
    cantidad_aristas += len(vecinos)
cantidad_aristas = cantidad_aristas // 2

print("Grafo no dirigido:")
print(f"  nodos   : {cantidad_nodos:,}")
print(f"  aristas : {cantidad_aristas:,}")

# Paso 2: ejecutar Louvain con una semilla fija.
comunidades = louvain(adj_no_dirigido, semilla=42)

# Renumerar 0, 1, 2, ... para que sea más fácil leer los resultados.
comunidades = renumerar_comunidades(comunidades)

# Paso 3: medir la calidad de la partición.
Q = modularidad(adj_no_dirigido, comunidades)

print()
print(f"Modularidad Q = {Q:.3f}")
print(f"Número de comunidades = {len(set(comunidades.values())):,}")

# Paso 4: mostrar las comunidades más grandes.
grupos = resumen_comunidades(comunidades, grafo.etiquetas)

print()
print("Comunidades más grandes:")
for posicion, (id_comunidad, nombres) in enumerate(
    sorted(grupos.items(), key=lambda par: -len(par[1]))[:10],
    start=1,
):
    primeros = ", ".join(sorted(nombres)[:5])
    if len(nombres) > 5:
        primeros += ", ..."
    print(f"  Comunidad {posicion:>2}: {len(nombres):>4} nodos | {primeros}")

# Paso 5: ver en qué comunidad cae cada exchange etiquetado.
print()
print("Comunidad de cada exchange etiquetado:")
for direccion, info in sorted(
    grafo.etiquetas.items(), key=lambda par: par[1]["nombre"]
):
    if direccion not in comunidades:
        continue
    id_comunidad = comunidades[direccion]
    tamano = len(grupos[id_comunidad])
    print(f"  {info['nombre']:<18} comunidad {id_comunidad:>3} ({tamano} nodos)")

### Experimento: ¿qué pasa si cambiamos la semilla?

Louvain es un algoritmo **heurístico**: el orden en que prueba los nodos
puede cambiar el resultado. Por eso se repite con varias semillas y se mira
si Q se mantiene estable.

In [ ]:
# ============ EXPERIMENTO: ESTABILIDAD CON SEMILLAS ============
import matplotlib.pyplot as plt

semillas = [1, 7, 42, 123, 2024]
resultados = []

for semilla in semillas:
    comunidades_semilla = louvain(adj_no_dirigido, semilla=semilla)
    q = modularidad(adj_no_dirigido, comunidades_semilla)
    cuantas = len(set(comunidades_semilla.values()))
    resultados.append((semilla, q, cuantas))

print(f"{'Semilla':>8} {'Q':>8} {'Comunidades':>12}")
print("-" * 32)
for semilla, q, cuantas in resultados:
    print(f"{semilla:>8} {q:>8.3f} {cuantas:>12}")

mejor = max(resultados, key=lambda fila: fila[1])
print()
print(f"Mejor semilla: {mejor[0]} con Q = {mejor[1]:.3f}")

plt.figure(figsize=(6, 4))
plt.bar(
    [str(fila[0]) for fila in resultados],
    [fila[1] for fila in resultados],
    color="seagreen",
)
plt.xlabel("Semilla")
plt.ylabel("Modularidad Q")
plt.title("Sensibilidad de Louvain a la semilla")
plt.ylim(0, 1)
plt.show()

## Tarea 4 — Kruskal: árbol de expansión mínima

**Responsabilidad:** grupal

**Estructura de datos:** lista de aristas + Union-Find.

Se ordenan las aristas por peso y se agregan solo si no forman un ciclo,
usando **Union-Find**. El resultado tiene `|V|-1` aristas.

In [ ]:
"""
arbol_expansion.py
==================
Tarea 4 — Árbol de expansión mínima (MST).

Problema: dado un grafo ponderado, encontrar el conjunto de aristas que
conecta TODOS los nodos con el MENOR costo total posible. Ese conjunto
siempre forma un árbol (no tiene ciclos) y tiene exactamente |V|-1 aristas.

Algoritmo elegido: Kruskal.

    1. Ordenar las aristas por peso de menor a mayor.
    2. Recorrerlas en orden y agregar cada arista SOLO si no forma un ciclo.
    3. Para detectar ciclos usamos Union-Find (conjuntos disjuntos).

Union-Find es una estructura de datos clásica que permite saber rápido si
dos nodos ya están conectados (están en el mismo conjunto) y unirlos.
"""

from typing import Dict, List, Set, Tuple


class UnionFind:
    """Conjuntos disjuntos con compresión de caminos y unión por rango."""

    def __init__(self, nodos: Set[str]) -> None:
        # Cada nodo apunta a su "padre". Al inicio, cada uno es su propio padre.
        self.padre: Dict[str, str] = {v: v for v in nodos}
        # El "rango" es una cota de la altura del árbol de cada conjunto.
        self.rango: Dict[str, int] = {v: 0 for v in nodos}

    def encontrar(self, x: str) -> str:
        """Devuelve el representante (raíz) del conjunto al que pertenece x."""
        # Compresión de caminos: hacemos que cada nodo apunte directo a la raíz,
        # así las siguientes búsquedas son más rápidas.
        if self.padre[x] != x:
            self.padre[x] = self.encontrar(self.padre[x])
        return self.padre[x]

    def unir(self, x: str, y: str) -> bool:
        """Une los conjuntos de x e y.

        Retorna:
            True si estaban en conjuntos distintos (y los unió),
            False si ya estaban conectados (agregar la arista crearía un ciclo).
        """
        rx = self.encontrar(x)
        ry = self.encontrar(y)
        if rx == ry:
            return False   # ya conectados: esta arista formaría un ciclo

        # Unión por rango: el árbol más "bajo" cuelga del más "alto".
        if self.rango[rx] < self.rango[ry]:
            rx, ry = ry, rx
        self.padre[ry] = rx
        if self.rango[rx] == self.rango[ry]:
            self.rango[rx] += 1
        return True


def aristas_no_dirigidas(aristas: List[Tuple[str, str, float]]) -> List[Tuple[str, str, float]]:
    """Convierte aristas dirigidas en no dirigidas.

    Si aparecen (a, b, w1) y (b, a, w2), las juntamos en una sola arista
    {a, b} con peso w1 + w2.
    """
    pesos: Dict[Tuple[str, str], float] = {}
    for a, b, w in aristas:
        clave = tuple(sorted((a, b)))
        pesos[clave] = pesos.get(clave, 0.0) + w
    return [(a, b, w) for (a, b), w in pesos.items()]


def kruskal(
    nodos: Set[str],
    aristas: List[Tuple[str, str, float]],
) -> Tuple[List[Tuple[str, str, float]], float]:
    """Construye el árbol de expansión mínima con el algoritmo de Kruskal.

    Parámetros:
        nodos: conjunto de todos los vértices.
        aristas: lista de aristas NO dirigidas (a, b, peso).

    Retorna:
        (mst, costo_total) donde mst es la lista de aristas del árbol.
    """
    # 1. Ordenar las aristas por peso, de menor a mayor.
    ordenadas = sorted(aristas, key=lambda e: e[2])

    # 2. Estructura Union-Find para detectar ciclos.
    uf = UnionFind(nodos)

    mst: List[Tuple[str, str, float]] = []
    costo = 0.0
    objetivo = len(nodos) - 1   # un árbol con |V| nodos tiene |V|-1 aristas

    # 3. Recorrer las aristas en orden.
    for a, b, w in ordenadas:
        if len(mst) == objetivo:
            break   # ya tenemos el árbol completo

        if uf.unir(a, b):
            mst.append((a, b, w))
            costo += w

    return mst, costo


def resumen_mst(mst: List[Tuple[str, str, float]]) -> Dict[str, float]:
    """Resumen simple del MST: número de aristas y costo total."""
    return {
        "aristas": len(mst),
        "costo_total": sum(w for _, _, w in mst),
    }


def kruskal_desde_aristas(
    aristas: List[Tuple[str, str, float]],
) -> Tuple[List[Tuple[str, str, float]], float]:
    """Kruskal a partir de la lista compartida de aristas.

    Convierte las aristas dirigidas en no dirigidas (sumando pesos si hay
    transferencias en ambos sentidos) y calcula el MST.
    """
    no_dirigidas = aristas_no_dirigidas(
        [(origen, destino, float(peso)) for origen, destino, peso in aristas]
    )
    nodos: Set[str] = set()
    for origen, destino, _ in no_dirigidas:
        nodos.add(origen)
        nodos.add(destino)
    return kruskal(nodos, no_dirigidas)


# ============ DEMO TAREA 4 ============
mst, costo = kruskal_desde_aristas(grafo.aristas)
print(f"MST: {len(mst)} aristas, costo total = {costo:,.2f}\n")
print("Primeras 20 aristas del MST:")
for a, b, w in sorted(mst, key=lambda e: e[2])[:20]:
    print(f"  {grafo.nombre(a):<18} -- {grafo.nombre(b):<18} {w:,.2f}")

## Estadísticas del dataset completo

Esta sección sí usa las **38.901.039 transferencias**. DuckDB lee el Parquet
por columnas y calcula los totales sin cargar todo en RAM.

In [ ]:
# ============ ANÁLISIS DEL DATASET COMPLETO ============
import os
import time

if not RUTA_PARQUET:
    print("Sección omitida: no encontré el Parquet completo.")
    print("Súbelo a Drive como MyDrive/ED2/transferencias.parquet.")
else:
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "pyarrow"], check=True)
        import duckdb

    con = duckdb.connect()
    con.execute("SET memory_limit='2GB'")
    con.execute("SET threads=2")
    con.execute("SET preserve_insertion_order=false")

    t0 = time.perf_counter()
    r = con.execute(f"""
        SELECT count(*) filas,
               count(DISTINCT token_address) tokens,
               min(block_number) bmin,
               max(block_number) bmax
        FROM read_parquet('{RUTA_PARQUET}')
    """).fetchone()
    print(f"Transferencias : {r[0]:,}")
    print(f"Tokens         : {r[1]}")
    print(f"Bloques        : {r[2]:,} .. {r[3]:,}")
    print(f"Tiempo         : {time.perf_counter()-t0:.1f} s")

    print("\nTop 10 tokens por transferencias:")
    for tok, n in con.execute(f"""
        SELECT token_address, count(*) n
        FROM read_parquet('{RUTA_PARQUET}')
        GROUP BY 1 ORDER BY n DESC LIMIT 10
    """).fetchall():
        print(f"  {tok}  {n:>10,}")

    print("\nActividad mensual aproximada:")
    for mes, n in con.execute(f"""
        SELECT strftime(to_timestamp(epoch(TIMESTAMP '2018-10-08') + (block_number-6525201)*13.5), '%Y-%m') mes,
               count(*) n
        FROM read_parquet('{RUTA_PARQUET}')
        GROUP BY mes ORDER BY mes
    """).fetchall():
        print(f"  {mes}: {n:>10,}")

## Conclusiones

1. **Un mismo dataset admite varias preguntas.** Bellman-Ford responde
   caminos, PageRank responde importancia, Louvain responde grupos y Kruskal
   responde conectividad mínima.

2. **Las estructuras de datos importan.** La lista de adyacencia, Union-Find y
   los diccionarios hacen que los algoritmos sean simples y legibles.

3. **La escala importa.** El análisis completo usa 38,9 M de transferencias con
   DuckDB; los algoritmos en Python puro se ejecutan sobre una muestra
   conectada de USDC.

4. **La muestra no reemplaza al dataset.** El Parquet conserva las 38.901.039
   filas; la muestra es solo para que los algoritmos terminen en Colab.

5. **Todo es reproducible.** La muestra tiene semilla 42 y el pipeline está
   documentado en el repositorio.